
<div style="background:linear-gradient(135deg,#1a2a4a 0%,#2d4a7a 100%);padding:40px;border-radius:12px;margin-bottom:30px;">
  <h1 style="color:white;font-size:2.2em;margin:0 0 10px 0;">🚀 Step 4 — FastAPI Backend</h1>
  <p style="color:#a8c4e0;font-size:1.1em;margin:0;">Supply Chain Planning System · Manufacturing Sector</p>
  <div style="margin-top:20px;display:flex;gap:15px;flex-wrap:wrap;">
    <span style="background:rgba(255,255,255,0.15);color:white;padding:5px 14px;border-radius:20px;font-size:.85em;">⚡ FastAPI</span>
    <span style="background:rgba(255,255,255,0.15);color:white;padding:5px 14px;border-radius:20px;font-size:.85em;">📦 Pydantic schemas</span>
    <span style="background:rgba(255,255,255,0.15);color:white;padding:5px 14px;border-radius:20px;font-size:.85em;">🔗 REST endpoints</span>
    <span style="background:rgba(255,255,255,0.15);color:white;padding:5px 14px;border-radius:20px;font-size:.85em;">🧪 Live testing</span>
  </div>
</div>



<div style="background:#f0f7ff;border-left:5px solid #2d6cdf;padding:20px 25px;border-radius:0 8px 8px 0;margin-bottom:25px;">
  <h2 style="color:#1a2a4a;margin-top:0;">🎯 Purpose of this notebook</h2>
  <p style="color:#333;line-height:1.7;">This notebook is different from the previous ones. Instead of data analysis, it <strong>generates the actual source files</strong> of the FastAPI backend and then tests every endpoint live.</p>
  <div style="background:#1e1e1e;color:#a8ff78;padding:15px 20px;border-radius:8px;font-family:monospace;font-size:.88em;line-height:1.9;margin:12px 0;">
    src/api/<br>
    ├── main.py&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; &lt;- FastAPI app entry point<br>
    ├── models.py&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&lt;- Pydantic request/response schemas<br>
    ├── routers/<br>
    │&nbsp;&nbsp; ├── forecast.py&nbsp;&nbsp;&lt;- /forecast endpoints<br>
    │&nbsp;&nbsp; └── alerts.py&nbsp;&nbsp;&nbsp; &lt;- /alerts endpoints<br>
    └── services/<br>
    &nbsp;&nbsp;&nbsp;&nbsp; └── predictor.py &lt;- model loading + inference
  </div>
</div>



<div style="background:#1a2a4a;padding:12px 20px;border-radius:8px;margin:20px 0 5px 0;">
  <h2 style="color:white;margin:0;font-size:1.2em;">🔧 Section 0 — Install dependencies & create folders</h2>
</div>
<div style="background:#f8f9fa;border:1px solid #dee2e6;padding:18px 22px;border-radius:8px;margin-top:8px;">
  <h4 style="color:#1a2a4a;margin-top:0;">📖 What this section does</h4>
  <p style="color:#444;line-height:1.7;">Installs FastAPI and its dependencies, then creates the folder structure. <code>uvicorn</code> is the ASGI server that actually runs the app. <code>httpx</code> is the HTTP client we use to test endpoints from within the notebook.</p>
</div>


In [6]:
import subprocess, sys, os

subprocess.run([sys.executable, '-m', 'pip', 'install',
                'fastapi', 'uvicorn[standard]', 'httpx', 'python-multipart',
                '--quiet'], check=True)
print('✅ FastAPI stack installed')

dirs = ['src/api', 'src/api/routers', 'src/api/services']
for d in dirs:
    os.makedirs(d, exist_ok=True)
    init = f'{d}/__init__.py'
    if not os.path.exists(init):
        open(init, 'w').close()
print('✅ Folder structure ready')
for d in dirs:
    print(f'   {d}/')


✅ FastAPI stack installed
✅ Folder structure ready
   src/api/
   src/api/routers/
   src/api/services/



<div style="background:#1a2a4a;padding:12px 20px;border-radius:8px;margin:20px 0 5px 0;">
  <h2 style="color:white;margin:0;font-size:1.2em;">📁 Section 1 — Write all API source files</h2>
</div>
<div style="background:#f8f9fa;border:1px solid #dee2e6;padding:18px 22px;border-radius:8px;margin-top:8px;">
  <h4 style="color:#1a2a4a;margin-top:0;">📖 What this section does</h4>
  <p style="color:#444;line-height:1.7;">Writes all 5 Python source files to disk. Each file is written with <code>pathlib.Path.write_text()</code> so you can inspect them in VS Code after running this cell. The files are real, editable Python modules.</p>
  <table style="width:100%;font-size:.9em;border-collapse:collapse;margin-top:10px;">
    <tr style="background:#2d6cdf;color:white;"><th style="padding:8px 12px;">File</th><th style="padding:8px 12px;">Responsibility</th></tr>
    <tr style="background:white;"><td style="padding:8px 12px;"><code>models.py</code></td><td style="padding:8px 12px;">Pydantic schemas - data contracts for every request and response</td></tr>
    <tr style="background:#f0f7ff;"><td style="padding:8px 12px;"><code>services/predictor.py</code></td><td style="padding:8px 12px;">ML logic - loads model, runs inference, generates SHAP and alerts</td></tr>
    <tr style="background:white;"><td style="padding:8px 12px;"><code>routers/forecast.py</code></td><td style="padding:8px 12px;">HTTP endpoints for demand forecasting</td></tr>
    <tr style="background:#f0f7ff;"><td style="padding:8px 12px;"><code>routers/alerts.py</code></td><td style="padding:8px 12px;">HTTP endpoints for supply chain alerts</td></tr>
    <tr style="background:white;"><td style="padding:8px 12px;"><code>main.py</code></td><td style="padding:8px 12px;">App entry point - wires everything together, configures CORS</td></tr>
  </table>
</div>


In [8]:
from pathlib import Path

# ── models.py ────────────────────────────────────────────────────────────────
# Pydantic BaseModel subclasses define the shape of every request and response.
# FastAPI uses these for: automatic validation, serialisation, and /docs generation.
Path('src/api/models.py').write_text(
'from pydantic import BaseModel, Field\n'
'from typing import Optional, List, Dict\n'
'\n'
'class HealthResponse(BaseModel):\n'
'    status: str\n'
'    model_loaded: bool\n'
'    n_features: int\n'
'    timestamp: str\n'
'\n'
'class ForecastRequest(BaseModel):\n'
'    product_id: int = Field(..., ge=1)\n'
'    week_index: int = Field(..., ge=0)\n'
'    features: Dict[str, float]\n'
'\n'
'class ForecastResponse(BaseModel):\n'
'    product_id: int\n'
'    week_index: int\n'
'    predicted_demand: float\n'
'    confidence_lower: float\n'
'    confidence_upper: float\n'
'    timestamp: str\n'
'\n'
'class BatchForecastRequest(BaseModel):\n'
'    requests: List[ForecastRequest]\n'
'\n'
'class BatchForecastResponse(BaseModel):\n'
'    forecasts: List[ForecastResponse]\n'
'    n_predictions: int\n'
'    timestamp: str\n'
'\n'
'class LatestForecastResponse(BaseModel):\n'
'    product_id: int\n'
'    predicted_demand: float\n'
'    recent_avg: float\n'
'    change_pct: float\n'
'    confidence_lower: float\n'
'    confidence_upper: float\n'
'\n'
'class Alert(BaseModel):\n'
'    product_id: int\n'
'    alert_type: str\n'
'    severity: str\n'
'    predicted: float\n'
'    recent_avg: float\n'
'    change_pct: float\n'
'    action: str\n'
'\n'
'class AlertsResponse(BaseModel):\n'
'    alerts: List[Alert]\n'
'    n_alerts: int\n'
'    week_index: int\n'
'    timestamp: str\n'
'\n'
'class ShapContribution(BaseModel):\n'
'    feature: str\n'
'    value: float\n'
'    shap_value: float\n'
'    direction: str\n'
'\n'
'class ShapResponse(BaseModel):\n'
'    product_id: int\n'
'    week_index: int\n'
'    prediction: float\n'
'    base_value: float\n'
'    top_contributions: List[ShapContribution]\n'
'    timestamp: str\n'
)
print('✅ src/api/models.py')


✅ src/api/models.py


In [9]:
# ── services/predictor.py ─────────────────────────────────────────────────────
# The bridge between HTTP layer and ML model.
# Loaded ONCE at startup and shared across all requests (singleton pattern).
# Key methods:
#   predict()              -> (float, float, float)  prediction + confidence bounds
#   predict_latest_week()  -> List[dict]             all products, latest data
#   explain()              -> dict                   SHAP contributions
#   generate_alerts()      -> (List[dict], int)      alerts + week index

predictor_src = '''
import pickle, json, os
import numpy as np
import pandas as pd
import shap
from datetime import datetime
from typing import Dict, List, Tuple

class PredictorService:
    def __init__(self,
                 model_path='models/xgb_demand_model.pkl',
                 features_path='data/processed/feature_list.json',
                 dataset_path='data/processed/ml_dataset.csv'):
        if not os.path.exists(model_path):
            raise FileNotFoundError(f'Model not found at {model_path}. Run Notebook 03 first.')
        with open(model_path, 'rb') as f:
            self.model = pickle.load(f)
        with open(features_path) as f:
            meta = json.load(f)
        self.features = meta['features']
        self.target   = meta['target']
        self.df       = pd.read_csv(dataset_path)
        self._explainer = None
        print(f'PredictorService ready - {len(self.features)} features')

    def predict(self, features: Dict[str, float]) -> Tuple[float, float, float]:
        import pandas as pd, numpy as np
        X    = pd.DataFrame([features])[self.features].fillna(0)
        pred = float(np.maximum(self.model.predict(X)[0], 0))
        return round(pred, 1), round(pred * 0.80, 1), round(pred * 1.20, 1)

    def predict_latest_week(self) -> List[Dict]:
        import numpy as np
        last_week = self.df['week_index'].max()
        latest    = self.df[self.df['week_index'] == last_week].copy()
        X         = latest[self.features].fillna(0)
        preds     = np.maximum(self.model.predict(X), 0)
        lag_cols  = [c for c in self.features if 'ordered_qty_lag_' in c
                     and int(c.split('_')[-1]) <= 4]
        results = []
        for i, (_, row) in enumerate(latest.iterrows()):
            pred       = float(preds[i])
            recent_avg = float(latest[lag_cols].iloc[i].mean()) if lag_cols else pred
            change_pct = (pred - recent_avg) / max(recent_avg, 1) * 100
            results.append({
                'product_id':       int(row['product_id']),
                'predicted_demand': round(pred, 1),
                'recent_avg':       round(recent_avg, 1),
                'change_pct':       round(change_pct, 1),
                'confidence_lower': round(pred * 0.80, 1),
                'confidence_upper': round(pred * 1.20, 1),
            })
        return results

    def explain(self, features: Dict[str, float], top_n: int = 10) -> Dict:
        import pandas as pd, numpy as np, shap
        if self._explainer is None:
            bg = self.df[self.features].fillna(0).sample(min(100, len(self.df)), random_state=42)
            self._explainer = shap.TreeExplainer(self.model, bg)
        X          = pd.DataFrame([features])[self.features].fillna(0)
        shap_vals  = self._explainer.shap_values(X)[0]
        base_value = float(self._explainer.expected_value)
        prediction = float(np.maximum(self.model.predict(X)[0], 0))
        contribs   = sorted(
            [{'feature': f, 'value': float(features.get(f, 0)),
              'shap_value': float(s), 'direction': 'UP' if s > 0 else 'DOWN'}
             for f, s in zip(self.features, shap_vals)],
            key=lambda x: abs(x['shap_value']), reverse=True)
        return {'prediction': round(prediction,1), 'base_value': round(base_value,1),
                'top_contributions': contribs[:top_n]}

    def generate_alerts(self) -> Tuple[List[Dict], int]:
        predictions = self.predict_latest_week()
        last_week   = int(self.df['week_index'].max())
        alerts = []
        for p in predictions:
            pred   = p['predicted_demand']
            recent = p['recent_avg']
            ratio  = pred / max(recent, 1)
            pid    = p['product_id']
            prod   = self.df[self.df['product_id'] == pid]
            risk   = float(prod['supply_risk_score'].iloc[-1]) if 'supply_risk_score' in prod.columns else 0.0
            if ratio > 1.30:
                alerts.append({'product_id': pid, 'alert_type': 'DEMAND_SPIKE',
                    'severity': 'HIGH' if ratio > 1.50 else 'MEDIUM',
                    'predicted': round(pred,1), 'recent_avg': round(recent,1),
                    'change_pct': round((ratio-1)*100,1),
                    'action': 'Increase production order / draw from safety stock'})
            elif ratio < 0.70:
                alerts.append({'product_id': pid, 'alert_type': 'DEMAND_DROP',
                    'severity': 'MEDIUM', 'predicted': round(pred,1),
                    'recent_avg': round(recent,1), 'change_pct': round((ratio-1)*100,1),
                    'action': 'Reduce production order / avoid overstock'})
            if risk > 0.6 and ratio >= 0.9:
                alerts.append({'product_id': pid, 'alert_type': 'SUPPLY_RISK',
                    'severity': 'HIGH', 'predicted': round(pred,1),
                    'recent_avg': round(recent,1), 'change_pct': round((ratio-1)*100,1),
                    'action': f'Supply risk={risk:.2f} - contact supplier proactively'})
        return alerts, last_week
'''

Path('src/api/services/predictor.py').write_text(predictor_src)
print('✅ src/api/services/predictor.py')


✅ src/api/services/predictor.py


In [10]:
# ── routers/forecast.py ───────────────────────────────────────────────────────
# Defines all /forecast/* HTTP endpoints.
# Uses Depends(get_predictor) to inject the shared PredictorService instance.
# Response models (response_model=...) tell FastAPI what JSON shape to return.

forecast_src = '''
from fastapi import APIRouter, Depends, HTTPException
from datetime import datetime
from typing import List
from ..models import (ForecastRequest, ForecastResponse, BatchForecastRequest,
                      BatchForecastResponse, LatestForecastResponse,
                      ShapResponse, ShapContribution)
from ..services.predictor import PredictorService

router = APIRouter(prefix='/forecast', tags=['Forecast'])
_predictor = None

def get_predictor():
    if _predictor is None:
        raise HTTPException(status_code=503, detail='Predictor not initialised')
    return _predictor

def set_predictor(p):
    global _predictor
    _predictor = p

@router.get('/latest', response_model=List[LatestForecastResponse])
async def get_latest_forecasts(predictor=Depends(get_predictor)):
    return predictor.predict_latest_week()

@router.get('/shap/{product_id}', response_model=ShapResponse)
async def get_shap(product_id: int, predictor=Depends(get_predictor)):
    prod = predictor.df[predictor.df['product_id'] == product_id]
    if len(prod) == 0:
        raise HTTPException(status_code=404, detail=f'Product {product_id} not found')
    row      = prod.iloc[-1]
    features = {f: float(row.get(f, 0)) for f in predictor.features}
    expl     = predictor.explain(features)
    return ShapResponse(product_id=product_id, week_index=int(row['week_index']),
        prediction=expl['prediction'], base_value=expl['base_value'],
        top_contributions=[ShapContribution(**c) for c in expl['top_contributions']],
        timestamp=datetime.utcnow().isoformat())

@router.get('/{product_id}', response_model=LatestForecastResponse)
async def get_product_forecast(product_id: int, predictor=Depends(get_predictor)):
    results = predictor.predict_latest_week()
    match   = [r for r in results if r['product_id'] == product_id]
    if not match:
        raise HTTPException(status_code=404, detail=f'Product {product_id} not found')
    return match[0]

@router.post('/single', response_model=ForecastResponse)
async def post_single_forecast(request: ForecastRequest, predictor=Depends(get_predictor)):
    pred, lower, upper = predictor.predict(request.features)
    return ForecastResponse(product_id=request.product_id, week_index=request.week_index,
        predicted_demand=pred, confidence_lower=lower, confidence_upper=upper,
        timestamp=datetime.utcnow().isoformat())

@router.post('/batch', response_model=BatchForecastResponse)
async def post_batch_forecast(request: BatchForecastRequest, predictor=Depends(get_predictor)):
    forecasts = []
    for req in request.requests:
        pred, lower, upper = predictor.predict(req.features)
        forecasts.append(ForecastResponse(product_id=req.product_id,
            week_index=req.week_index, predicted_demand=pred,
            confidence_lower=lower, confidence_upper=upper,
            timestamp=datetime.utcnow().isoformat()))
    return BatchForecastResponse(forecasts=forecasts, n_predictions=len(forecasts),
        timestamp=datetime.utcnow().isoformat())
'''

Path('src/api/routers/forecast.py').write_text(forecast_src)
print('✅ src/api/routers/forecast.py')


✅ src/api/routers/forecast.py


In [11]:
# ── routers/alerts.py ─────────────────────────────────────────────────────────
# Two endpoints:
#   GET /alerts          -> full list of alerts, optional ?severity=HIGH filter
#   GET /alerts/summary  -> KPI counts (total, by_type, by_severity)

alerts_src = '''
from fastapi import APIRouter, Depends, Query
from datetime import datetime
from ..models import AlertsResponse, Alert
from ..services.predictor import PredictorService
from .forecast import get_predictor

router = APIRouter(prefix='/alerts', tags=['Alerts'])

@router.get('', response_model=AlertsResponse)
async def get_alerts(severity: str = Query(default=None), predictor=Depends(get_predictor)):
    alerts_raw, week_idx = predictor.generate_alerts()
    if severity:
        alerts_raw = [a for a in alerts_raw if a['severity'] == severity.upper()]
    return AlertsResponse(alerts=[Alert(**a) for a in alerts_raw],
        n_alerts=len(alerts_raw), week_index=week_idx,
        timestamp=datetime.utcnow().isoformat())

@router.get('/summary')
async def get_summary(predictor=Depends(get_predictor)):
    alerts_raw, week_idx = predictor.generate_alerts()
    return {'week_index': week_idx, 'total_alerts': len(alerts_raw),
        'by_type': {
            'DEMAND_SPIKE': sum(1 for a in alerts_raw if a['alert_type']=='DEMAND_SPIKE'),
            'DEMAND_DROP':  sum(1 for a in alerts_raw if a['alert_type']=='DEMAND_DROP'),
            'SUPPLY_RISK':  sum(1 for a in alerts_raw if a['alert_type']=='SUPPLY_RISK'),
        },
        'by_severity': {
            'HIGH':   sum(1 for a in alerts_raw if a['severity']=='HIGH'),
            'MEDIUM': sum(1 for a in alerts_raw if a['severity']=='MEDIUM'),
        },
        'timestamp': datetime.utcnow().isoformat()}
'''

Path('src/api/routers/alerts.py').write_text(alerts_src)
print('✅ src/api/routers/alerts.py')


✅ src/api/routers/alerts.py


In [12]:
# ── main.py ───────────────────────────────────────────────────────────────────
# Entry point. Key concepts:
# - lifespan: loads model ONCE at startup, keeps it in memory for all requests
# - CORSMiddleware: allows React (localhost:3000) to call this API from the browser
# - include_router: registers /forecast and /alerts route groups

main_src = '''
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from contextlib import asynccontextmanager
from datetime import datetime
from .routers import forecast as forecast_module
from .routers import alerts as alerts_module
from .routers.forecast import set_predictor
from .services.predictor import PredictorService
from .models import HealthResponse

@asynccontextmanager
async def lifespan(app: FastAPI):
    print('Starting Supply Chain API...')
    predictor = PredictorService()
    set_predictor(predictor)
    print('Model loaded - API ready')
    yield
    print('Shutting down API')

app = FastAPI(
    title='Supply Chain Planning API',
    description='XGBoost demand forecasting for manufacturing supply chain.',
    version='1.0.0',
    lifespan=lifespan,
)

app.add_middleware(CORSMiddleware,
    allow_origins=['http://localhost:3000', 'http://localhost:5173'],
    allow_credentials=True, allow_methods=['*'], allow_headers=['*'])

app.include_router(forecast_module.router)
app.include_router(alerts_module.router)

@app.get('/', response_model=HealthResponse, tags=['Health'])
async def health_check():
    from .routers.forecast import _predictor
    return HealthResponse(status='ok', model_loaded=_predictor is not None,
        n_features=len(_predictor.features) if _predictor else 0,
        timestamp=datetime.utcnow().isoformat())
'''

Path('src/api/main.py').write_text(main_src)
print('✅ src/api/main.py')
print('\n✅ All 5 API files written. Open them in VS Code to inspect the code.')


✅ src/api/main.py

✅ All 5 API files written. Open them in VS Code to inspect the code.



<div style="background:#1a2a4a;padding:12px 20px;border-radius:8px;margin:20px 0 5px 0;">
  <h2 style="color:white;margin:0;font-size:1.2em;">🧪 Section 2 — Start server & test all endpoints</h2>
</div>
<div style="background:#f8f9fa;border:1px solid #dee2e6;padding:18px 22px;border-radius:8px;margin-top:8px;">
  <h4 style="color:#1a2a4a;margin-top:0;">📖 What this section does</h4>
  <p style="color:#444;line-height:1.7;">Starts the FastAPI server as a background subprocess, waits for it to load the model, then tests every endpoint using <code>httpx</code>. All from inside the notebook.</p>
  <p style="color:#666;font-size:.9em;margin-bottom:0;">⚠️ Run the <strong>shutdown cell</strong> at the bottom when done. Only one server can run on port 8000 at a time.</p>
</div>


In [3]:
import subprocess, time, sys

# Start uvicorn as background process
# --reload: restarts when source files change (development mode only)
server_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'src.api.main:app',
     '--port', '8000', '--reload'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

print('Starting server', end='')
for _ in range(8):
    time.sleep(1)
    print('.', end='', flush=True)
print(' done!')
print(f'   PID:  {server_process.pid}')
print(f'   URL:  http://localhost:8000')
print(f'   Docs: http://localhost:8000/docs')


Starting server........ done!
   PID:  11564
   URL:  http://localhost:8000
   Docs: http://localhost:8000/docs


In [13]:
import httpx, json
BASE = 'http://localhost:8000'

def show(label, r):
    icon = 'OK' if r.status_code == 200 else 'ERROR'
    print(f'\n[{icon}] {label}  (HTTP {r.status_code})')
    try:
        data  = r.json()
        lines = json.dumps(data, indent=2).split('\n')
        shown = lines[:25]
        print('\n'.join('   ' + l for l in shown))
        if len(lines) > 25:
            print(f'   ... [{len(lines)-25} more lines]')
    except Exception:
        print(f'   {r.text[:200]}')

# Test 1: Health
show('GET /  (health check)', httpx.get(f'{BASE}/'))



[OK] GET /  (health check)  (HTTP 200)
   {
     "status": "ok",
     "model_loaded": true,
     "n_features": 70,
     "timestamp": "2026-05-28T18:23:59.802550"
   }


In [14]:
# Test 2: All forecasts
show('GET /forecast/latest  (all products)', httpx.get(f'{BASE}/forecast/latest'))



[OK] GET /forecast/latest  (all products)  (HTTP 200)
   [
     {
       "product_id": 1,
       "predicted_demand": 58.1,
       "recent_avg": 296.0,
       "change_pct": -80.4,
       "confidence_lower": 46.5,
       "confidence_upper": 69.7
     },
     {
       "product_id": 2,
       "predicted_demand": 36.1,
       "recent_avg": 144.7,
       "change_pct": -75.0,
       "confidence_lower": 28.9,
       "confidence_upper": 43.3
     },
     {
       "product_id": 3,
       "predicted_demand": 128.5,
       "recent_avg": 665.3,
       "change_pct": -80.7,
       "confidence_lower": 102.8,
       "confidence_upper": 154.2
     },
   ... [57 more lines]


In [15]:
# Test 3: Single product
show('GET /forecast/3  (product 3)', httpx.get(f'{BASE}/forecast/3'))



[OK] GET /forecast/3  (product 3)  (HTTP 200)
   {
     "product_id": 3,
     "predicted_demand": 128.5,
     "recent_avg": 665.3,
     "change_pct": -80.7,
     "confidence_lower": 102.8,
     "confidence_upper": 154.2
   }


In [16]:
# Test 4: All alerts
show('GET /alerts  (all active alerts)', httpx.get(f'{BASE}/alerts'))



[OK] GET /alerts  (all active alerts)  (HTTP 200)
   {
     "alerts": [
       {
         "product_id": 1,
         "alert_type": "DEMAND_DROP",
         "severity": "MEDIUM",
         "predicted": 58.1,
         "recent_avg": 296.0,
         "change_pct": -80.4,
         "action": "Reduce production order / avoid overstock"
       },
       {
         "product_id": 2,
         "alert_type": "DEMAND_DROP",
         "severity": "MEDIUM",
         "predicted": 36.1,
         "recent_avg": 144.7,
         "change_pct": -75.1,
         "action": "Reduce production order / avoid overstock"
       },
       {
         "product_id": 3,
         "alert_type": "DEMAND_DROP",
         "severity": "MEDIUM",
         "predicted": 128.5,
   ... [72 more lines]


In [17]:
# Test 5: Alerts summary (KPI cards)
show('GET /alerts/summary  (KPI counts)', httpx.get(f'{BASE}/alerts/summary'))



[OK] GET /alerts/summary  (KPI counts)  (HTTP 200)
   {
     "week_index": 102,
     "total_alerts": 10,
     "by_type": {
       "DEMAND_SPIKE": 0,
       "DEMAND_DROP": 10,
       "SUPPLY_RISK": 0
     },
     "by_severity": {
       "HIGH": 0,
       "MEDIUM": 10
     },
     "timestamp": "2026-05-28T18:24:24.085988"
   }


In [18]:
# Test 6: SHAP explanation
show('GET /forecast/shap/1  (SHAP for product 1)', httpx.get(f'{BASE}/forecast/shap/1'))



[OK] GET /forecast/shap/1  (SHAP for product 1)  (HTTP 200)
   {
     "product_id": 1,
     "week_index": 102,
     "prediction": 58.1,
     "base_value": 218.9,
     "top_contributions": [
       {
         "feature": "week_index",
         "value": 102.0,
         "shap_value": -42.14861366401426,
         "direction": "DOWN"
       },
       {
         "feature": "ordered_qty_lag_52",
         "value": 91.0,
         "shap_value": -40.84494203399867,
         "direction": "DOWN"
       },
       {
         "feature": "pmi",
         "value": 50.53,
         "shap_value": -24.060416890177876,
         "direction": "DOWN"
       },
       {
   ... [44 more lines]



<div style="background:#fff3cd;border:1px solid #ffc107;padding:15px 20px;border-radius:8px;margin:15px 0;">
  <h4 style="color:#1a2a4a;margin-top:0;">🌐 Interactive docs</h4>
  <p style="color:#444;line-height:1.7;margin-bottom:0;">While the server is running, open <strong><a href="http://localhost:8000/docs" target="_blank">http://localhost:8000/docs</a></strong> in your browser. FastAPI generates a full Swagger UI where you can explore and call every endpoint interactively without writing any code.</p>
</div>


In [ ]:
# Run this when done testing
server_process.terminate()
server_process.wait()
print('Server stopped.')
print('To restart: re-run the Start server cell above')
print('To run standalone from terminal:')
print('   uvicorn src.api.main:app --reload --port 8000')



<div style="background:linear-gradient(135deg,#1a2a4a 0%,#2d4a7a 100%);padding:30px 35px;border-radius:12px;margin-top:30px;">
  <h2 style="color:white;margin-top:0;">✅ Step 4 complete — What we built</h2>
  <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:20px;margin-top:20px;">
    <div style="background:rgba(255,255,255,0.1);padding:15px;border-radius:8px;">
      <div style="color:#a8c4e0;font-size:.85em;margin-bottom:5px;">API FILES</div>
      <div style="color:white;font-size:1.4em;font-weight:bold;">5 modules</div>
      <div style="color:#a8c4e0;font-size:.8em;">main · models · predictor<br>forecast router · alerts router</div>
    </div>
    <div style="background:rgba(255,255,255,0.1);padding:15px;border-radius:8px;">
      <div style="color:#a8c4e0;font-size:.85em;margin-bottom:5px;">ENDPOINTS</div>
      <div style="color:white;font-size:1.4em;font-weight:bold;">7 endpoints</div>
      <div style="color:#a8c4e0;font-size:.8em;">health · /forecast/latest<br>/forecast/{id} · /forecast/shap/{id}<br>/forecast/single · /forecast/batch<br>/alerts · /alerts/summary</div>
    </div>
    <div style="background:rgba(255,255,255,0.1);padding:15px;border-radius:8px;">
      <div style="color:#a8c4e0;font-size:.85em;margin-bottom:5px;">NEXT STEP</div>
      <div style="color:white;font-size:1.4em;font-weight:bold;">Notebook 05</div>
      <div style="color:#a8c4e0;font-size:.8em;">React dashboard<br>Charts · alerts · SHAP UI</div>
    </div>
  </div>
  <div style="margin-top:20px;padding:15px;background:rgba(255,255,255,0.08);border-radius:8px;">
    <p style="color:#a8c4e0;margin:0;font-size:.9em;">Run standalone: <code style="color:white;">uvicorn src.api.main:app --reload --port 8000</code></p>
  </div>
</div>


In [3]:
import os
print(os.getcwd())

c:\Users\paran\Downloads


In [4]:
import os
os.chdir(r"C:\Users\paran\OneDrive\python_projects\supply-chain-app")
print(os.getcwd())

C:\Users\paran\OneDrive\python_projects\supply-chain-app
